# Audit des datasets baoulé — Waxal + Common Voice

Ce notebook exécute uniquement l'audit préalable. Il ne lance aucun entraînement et ne modifie pas les datasets distants.

Sorties principales : résumé des splits, inventaire Unicode, doublons et liste des fichiers à écouter manuellement.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets[audio]>=4.0,<5", "numpy>=1.26,<3", "soundfile>=0.12,<1",
])
print("Dépendances d'audit installées.")

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/ngangoula-jaures/Finetune_omniASR_CTC_1B_for_baoule.git"
PROJECT_DIR = Path("/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule")

if not PROJECT_DIR.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / "src" / "audit_datasets.py").is_file():
    raise FileNotFoundError("src/audit_datasets.py est introuvable. Vérifie que les nouveaux fichiers ont été poussés.")

print("Projet :", PROJECT_DIR)

## Smoke test

Deux exemples par split sont d'abord décodés. Ne lance pas l'audit complet si cette cellule échoue.

In [ ]:
import os

SMOKE_OUTPUT = Path("/kaggle/working/baoule_ctc1b_audit_smoke")
smoke_command = [
    sys.executable, str(PROJECT_DIR / "src" / "audit_datasets.py"),
    "--output-dir", str(SMOKE_OUTPUT),
    "--cache-dir", "/kaggle/working/hf-cache",
    "--max-examples-per-split", "2",
]
subprocess.run(smoke_command, cwd=PROJECT_DIR, check=True, env={**os.environ, "HF_HOME": "/kaggle/working/hf-cache"})
print((SMOKE_OUTPUT / "dataset_summary.json").read_text(encoding="utf-8"))

## Audit complet

Cette cellule traite les trois splits des deux sources. Les tests restent séparés et ne sont jamais ajoutés à l'entraînement.

In [ ]:
AUDIT_OUTPUT = Path("/kaggle/working/baoule_ctc1b_dataset_audit")
audit_command = [
    sys.executable, str(PROJECT_DIR / "src" / "audit_datasets.py"),
    "--output-dir", str(AUDIT_OUTPUT),
    "--cache-dir", "/kaggle/working/hf-cache",
]
subprocess.run(audit_command, cwd=PROJECT_DIR, check=True, env={**os.environ, "HF_HOME": "/kaggle/working/hf-cache"})

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = json.loads((AUDIT_OUTPUT / "dataset_summary.json").read_text(encoding="utf-8"))
display(pd.DataFrame(summary["groups"]).T)
print("Totaux :", json.dumps(summary["totals"], ensure_ascii=False, indent=2))

review = pd.read_csv(AUDIT_OUTPUT / "manual_review.csv")
display(review[[
    "source", "split", "example_id", "speaker_id",
    "duration_seconds", "words_per_second", "status",
    "preprocessing_actions", "review_reasons",
    "text_raw", "text_normalized",
]].head(100))
print(f"{len(review)} exemples nécessitent une revue.")

## Sauvegarde

Télécharge le dossier `/kaggle/working/baoule_ctc1b_dataset_audit` ou sauvegarde-le comme sortie Kaggle. Ces fichiers seront analysés avant d'écrire la préparation MixtureParquet et de lancer le smoke test du modèle 1B.